In [1]:
import pandas as pd
from pathlib import Path

In [3]:
standardised_pay_gap = pd.read_csv(
    r"C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics\processed\standardised_pay_gap.csv")
standardised_pay_gap.shape

(49880, 20)

In [4]:
# Analysis A 

def summarise_pay(group_columns):
    return (
        standardised_pay_gap
        .groupby(group_columns, dropna=False, observed=True)
        .agg(
            headcount=("employee_id", "size"),
            mean_base_pay=("annual_base_pay_gbp", "mean"),
            median_base_pay=("annual_base_pay_gbp", "median"),
            mean_total_pay=("calculated_total_annual_pay_gbp", "mean"),
            median_total_pay=("calculated_total_annual_pay_gbp", "median"),
        )
        .reset_index()
    )

department_pay_summary = summarise_pay(["department", "gender"])
display(department_pay_summary.head(10))

,department,gender,headcount,mean_base_pay,median_base_pay,mean_total_pay,median_total_pay
0,Data,Female,2403,46269.288390,42900.0,50197.775110,46517.860
1,Data,Male,2428,46441.021417,43050.0,50460.440099,46908.830
2,Data,Non-binary,58,46546.551724,42750.0,50912.077414,47126.380
3,Data,Not recorded,20,49025.000000,46200.0,52656.974500,47503.305
4,Data,Prefer not to say,79,47741.772152,45600.0,51506.406835,48415.820
5,Engineering,Female,2425,49296.948454,46300.0,53475.533546,49676.420
6,Engineering,Male,2435,49376.591376,46000.0,53494.106275,50014.490
7,Engineering,Non-binary,53,56449.056604,51200.0,62158.775094,54623.270
8,Engineering,Not recorded,19,45436.842105,37800.0,48258.497368,40552.210
9,Engineering,Prefer not to say,73,45175.342466,42600.0,49155.754110,46363.860


In [5]:
overall_pay_summary = summarise_pay(["gender"])
job_level_pay_summary = summarise_pay(["job_level", "gender"])
employment_type_pay_summary = summarise_pay(["employment_type", "gender"])
work_location_pay_summary = summarise_pay(["work_location", "gender"])

display(overall_pay_summary)

,gender,headcount,mean_base_pay,median_base_pay,mean_total_pay,median_total_pay
0,Female,24182,42062.401786,38700.0,45632.983337,41938.195
1,Male,24074,41977.967932,38600.0,45537.429757,41930.065
2,Non-binary,597,41474.036851,38100.0,45114.215159,41772.680
3,Not recorded,160,41571.875000,37350.0,44846.987375,40776.370
4,Prefer not to say,867,40181.891580,37300.0,43591.358293,40441.370


In [7]:
years = standardised_pay_gap["years_at_company"]

standardised_pay_gap["tenure_band"] = pd.Series(
    pd.NA, index=standardised_pay_gap.index, dtype="string"
)

standardised_pay_gap.loc[
    (years >= 0) & (years < 2), "tenure_band"
] = "Under 2 years"

standardised_pay_gap.loc[
    (years >= 2) & (years < 5), "tenure_band"
] = "2 to under 5 years"

standardised_pay_gap.loc[
    (years >= 5) & (years < 10), "tenure_band"
] = "5 to under 10 years"

standardised_pay_gap.loc[
    years >= 10, "tenure_band"
] = "10+ years"

In [8]:
standardised_pay_gap["tenure_band"].value_counts(dropna=False)

tenure_band
10+ years              27341
5 to under 10 years    11321
2 to under 5 years      6820
Under 2 years           4398
Name: count, dtype: Int64

In [9]:
pay_columns = [
    "annual_base_pay_gbp",
    "annual_bonus_gbp",
    "calculated_total_annual_pay_gbp",
]

display(standardised_pay_gap[pay_columns].head())

,annual_base_pay_gbp,annual_bonus_gbp,calculated_total_annual_pay_gbp
0,63500,7121.05,70621.05
1,26800,1924.97,28724.97
2,35200,5334.13,40534.13
3,51500,2349.34,53849.34
4,35200,2306.93,37506.93


In [11]:
minimum_headcount = 10

#  Summarise annual base pay by department and gender.
department_summary = (
    standardised_pay_gap
    .groupby(["department", "gender"], dropna=False, observed=True)
    .agg(
        headcount=("employee_id", "size"),
        base_pay_count=("annual_base_pay_gbp", "count"),
        mean_base_pay=("annual_base_pay_gbp", "mean"),
        median_base_pay=("annual_base_pay_gbp", "median"),
    )
    .reset_index()
)

#  Put each group's pay beside the Male pay in the same department.
is_male = department_summary["gender"].eq("Male")
male_reference = department_summary.loc[is_male].drop(columns="gender")

department_pay_gaps = department_summary.loc[~is_male].merge(
    male_reference,
    on="department",
    how="left",
    suffixes=("", "_male"),
    validate="many_to_one",
)

# Show zero Male employees where a department has no Male reference group.
male_count_columns = ["headcount_male", "base_pay_count_male"]
department_pay_gaps[male_count_columns] = (
    department_pay_gaps[male_count_columns].fillna(0).astype(int)
)

#  Apply the minimum count and calculate the gaps safely.
enough_employees = (
    department_pay_gaps["base_pay_count"].ge(minimum_headcount)
    & department_pay_gaps["base_pay_count_male"].ge(minimum_headcount)
)

male_mean = department_pay_gaps["mean_base_pay_male"]
male_median = department_pay_gaps["median_base_pay_male"]

department_pay_gaps["mean_base_pay_gap_pct"] = (
    (male_mean - department_pay_gaps["mean_base_pay"])
    .div(male_mean.where(enough_employees & male_mean.ne(0)))
    .mul(100)
)

department_pay_gaps["median_base_pay_gap_pct"] = (
    (male_median - department_pay_gaps["median_base_pay"])
    .div(male_median.where(enough_employees & male_median.ne(0)))
    .mul(100)
)

display(department_pay_gaps[[
    "department", "gender",
    "headcount", "headcount_male",
    "base_pay_count", "base_pay_count_male",
    "mean_base_pay_gap_pct", "median_base_pay_gap_pct",
]].round(2))

,department,gender,headcount,headcount_male,base_pay_count,base_pay_count_male,mean_base_pay_gap_pct,median_base_pay_gap_pct
0,Data,Female,2403,2428,2403,2428,0.37,0.35
1,Data,Non-binary,58,2428,58,2428,-0.23,0.70
2,Data,Not recorded,20,2428,20,2428,-5.56,-7.32
3,Data,Prefer not to say,79,2428,79,2428,-2.80,-5.92
4,Engineering,Female,2425,2435,2425,2435,0.16,-0.65
5,Engineering,Non-binary,53,2435,53,2435,-14.32,-11.30
6,Engineering,Not recorded,19,2435,19,2435,7.98,17.83
7,Engineering,Prefer not to say,73,2435,73,2435,8.51,7.39
8,Finance,Female,2456,2369,2456,2369,-0.47,-0.51
9,Finance,Non-binary,58,2369,58,2369,7.73,9.97


In [12]:
def calculate_pay_gaps(data, group_column, pay_column):
    summary = (
        data.groupby([group_column, "gender"], observed=True, dropna=False)
        .agg(
            headcount=("employee_id", "size"),
            pay_count=(pay_column, "count"),
            mean_pay=(pay_column, "mean"),
            median_pay=(pay_column, "median"),
        )
        .reset_index()
    )

    is_male = summary["gender"].eq("Male")
    male_reference = summary.loc[is_male].drop(columns="gender")

    gaps = summary.loc[~is_male].merge(
        male_reference,
        on=group_column,
        how="left",
        suffixes=("", "_male"),
        validate="many_to_one",
    )

    male_count_columns = ["headcount_male", "pay_count_male"]
    gaps[male_count_columns] = gaps[male_count_columns].fillna(0).astype(int)

    enough_employees = gaps["pay_count"].ge(10) & gaps["pay_count_male"].ge(10)

    male_mean = gaps["mean_pay_male"]
    male_median = gaps["median_pay_male"]

    gaps["mean_gap_pct"] = (
        (male_mean - gaps["mean_pay"])
        .div(male_mean.where(enough_employees & male_mean.ne(0)))
        .mul(100)
    )

    gaps["median_gap_pct"] = (
        (male_median - gaps["median_pay"])
        .div(male_median.where(enough_employees & male_median.ne(0)))
        .mul(100)
    )

    return gaps.rename(columns={group_column: "group"}).assign(
        breakdown=group_column
    )

In [13]:
pay_data = standardised_pay_gap.assign(overall="Organisation")
breakdowns = ["overall", "job_level", "department"]

base_pay_comparison = pd.concat(
    [
        calculate_pay_gaps(pay_data, breakdown, "annual_base_pay_gbp")
        for breakdown in breakdowns
    ],
    ignore_index=True,
)

total_pay_comparison = pd.concat(
    [
        calculate_pay_gaps(pay_data, breakdown, "calculated_total_annual_pay_gbp")
        for breakdown in breakdowns
    ],
    ignore_index=True,
)

In [14]:
comparison_columns = [
    "breakdown", "group",
    "headcount", "headcount_male",
    "pay_count", "pay_count_male",
    "mean_gap_pct", "median_gap_pct",
]

display(
    base_pay_comparison.loc[
        base_pay_comparison["gender"].eq("Female"), comparison_columns
    ].round(2)
)

,breakdown,group,headcount,headcount_male,pay_count,pay_count_male,mean_gap_pct,median_gap_pct
0,overall,Organisation,24182,24074,24182,24074,-0.20,-0.26
4,job_level,1,6090,5962,6090,5962,-0.62,-0.37
8,job_level,2,7655,7753,7655,7753,-0.10,-0.28
12,job_level,3,5564,5500,5564,5500,0.21,0.65
16,job_level,4,3355,3437,3355,3437,-0.27,-0.34
20,job_level,5,1518,1422,1518,1422,0.03,-0.27
24,department,Data,2403,2428,2403,2428,0.37,0.35
28,department,Engineering,2425,2435,2425,2435,0.16,-0.65
32,department,Finance,2456,2369,2456,2369,-0.47,-0.51
36,department,HR,2345,2425,2345,2425,-2.41,-2.66


In [15]:
pay_data = standardised_pay_gap.assign(overall="Organisation")

breakdowns = [
    "overall",
    "job_level",
    "department",
    "employment_type",
    "work_location",
    "tenure_band",
]

base_pay_comparison = pd.concat(
    [
        calculate_pay_gaps(pay_data, breakdown, "annual_base_pay_gbp")
        for breakdown in breakdowns
    ],
    ignore_index=True,
)

total_pay_comparison = pd.concat(
    [
        calculate_pay_gaps(pay_data, breakdown, "calculated_total_annual_pay_gbp")
        for breakdown in breakdowns
    ],
    ignore_index=True,
)

In [17]:
new_breakdowns = ["employment_type", "work_location", "tenure_band"]

columns_to_show = [
    "breakdown", "group", "gender",
    "headcount", "headcount_male",
    "pay_count", "pay_count_male",
    "mean_gap_pct", "median_gap_pct",
]

print("Annual base-pay gaps")
display(
    base_pay_comparison.loc[
        base_pay_comparison["breakdown"].isin(new_breakdowns), columns_to_show
    ].round(2)
)

print("Total-pay gaps")
display(
    total_pay_comparison.loc[
        total_pay_comparison["breakdown"].isin(new_breakdowns), columns_to_show
    ].round(2)
)

Annual base-pay gaps


,breakdown,group,gender,headcount,headcount_male,pay_count,pay_count_male,mean_gap_pct,median_gap_pct
64,employment_type,Fixed-term,Female,8079,8098,8079,8098,-0.56,-0.52
65,employment_type,Fixed-term,Non-binary,231,8098,231,8098,-1.02,-2.08
66,employment_type,Fixed-term,Not recorded,63,8098,63,8098,-1.41,1.56
67,employment_type,Fixed-term,Prefer not to say,293,8098,293,8098,4.60,2.60
68,employment_type,Part-time,Female,8054,8029,8054,8029,0.05,-0.78
69,employment_type,Part-time,Non-binary,182,8029,182,8029,1.63,0.90
70,employment_type,Part-time,Not recorded,40,8029,40,8029,7.10,5.43
71,employment_type,Part-time,Prefer not to say,286,8029,286,8029,3.41,4.01
72,employment_type,Permanent,Female,8049,7947,8049,7947,-0.09,0.26
73,employment_type,Permanent,Non-binary,184,7947,184,7947,3.46,3.75


Total-pay gaps


,breakdown,group,gender,headcount,headcount_male,pay_count,pay_count_male,mean_gap_pct,median_gap_pct
64,employment_type,Fixed-term,Female,8079,8098,8079,8098,-0.61,-0.66
65,employment_type,Fixed-term,Non-binary,231,8098,231,8098,-1.42,-3.75
66,employment_type,Fixed-term,Not recorded,63,8098,63,8098,-0.64,1.74
67,employment_type,Fixed-term,Prefer not to say,293,8098,293,8098,4.69,2.14
68,employment_type,Part-time,Female,8054,8029,8054,8029,0.05,-0.40
69,employment_type,Part-time,Non-binary,182,8029,182,8029,1.88,0.14
70,employment_type,Part-time,Not recorded,40,8029,40,8029,7.90,7.13
71,employment_type,Part-time,Prefer not to say,286,8029,286,8029,3.60,4.87
72,employment_type,Permanent,Female,8049,7947,8049,7947,-0.07,0.95
73,employment_type,Permanent,Non-binary,184,7947,184,7947,2.83,3.40


In [19]:
minimum_headcount = 10

bonus_data = pay_data.copy()
bonus_data["received_bonus"] = bonus_data["annual_bonus_gbp"].gt(0)
bonus_data["recipient_bonus_gbp"] = bonus_data["annual_bonus_gbp"].where(
    bonus_data["received_bonus"]
)

def summarise_bonus(group_column):
    summary = (
        bonus_data.groupby([group_column, "gender"], dropna=False, observed=True)
        .agg(
            headcount=("employee_id", "size"),
            bonus_recipients=("received_bonus", "sum"),
            average_recipient_bonus_gbp=("recipient_bonus_gbp", "mean"),
        )
        .reset_index()
    )

    eligible_headcount = summary["headcount"].where(
        summary["headcount"].ge(minimum_headcount)
    )
    summary["bonus_recipient_rate_pct"] = (
        summary["bonus_recipients"].div(eligible_headcount).mul(100)
    )
    summary["average_recipient_bonus_gbp"] = (
        summary["average_recipient_bonus_gbp"].where(
            summary["bonus_recipients"].ge(minimum_headcount)
        )
    )

    return summary.rename(columns={group_column: "group"}).assign(
        breakdown=group_column
    )

bonus_breakdowns = [
    "overall", "department", "job_level",
    "employment_type", "work_location", "tenure_band",
]

bonus_summary = pd.concat(
    [summarise_bonus(column) for column in bonus_breakdowns],
    ignore_index=True,
)

display(bonus_summary[[
    "breakdown", "group", "gender", "headcount", "bonus_recipients",
    "bonus_recipient_rate_pct", "average_recipient_bonus_gbp",
]].round(2))

,breakdown,group,gender,headcount,bonus_recipients,bonus_recipient_rate_pct,average_recipient_bonus_gbp
0,overall,Organisation,Female,24182,24182,100.0,3570.58
1,overall,Organisation,Male,24074,24074,100.0,3559.46
2,overall,Organisation,Non-binary,597,597,100.0,3640.18
3,overall,Organisation,Not recorded,160,160,100.0,3275.11
4,overall,Organisation,Prefer not to say,867,867,100.0,3409.47
...,...,...,...,...,...,...,...
150,tenure_band,Under 2 years,Female,2131,2131,100.0,3516.54
151,tenure_band,Under 2 years,Male,2107,2107,100.0,3529.85
152,tenure_band,Under 2 years,Non-binary,50,50,100.0,3682.23
153,tenure_band,Under 2 years,Not recorded,14,14,100.0,3637.93


In [20]:
# calculating gender representation 

def summarise_representation(group_column):
    summary = (
        standardised_pay_gap
        .groupby([group_column, "gender"], dropna=False, observed=True)
        .size()
        .rename("gender_headcount")
        .reset_index()
    )

    summary["group_headcount"] = (
        summary.groupby(group_column, dropna=False, observed=True)
        ["gender_headcount"].transform("sum")
    )
    summary["gender_share_pct"] = (
        summary["gender_headcount"].div(summary["group_headcount"]).mul(100)
    )

    return summary

department_representation = summarise_representation("department")
job_level_representation = summarise_representation("job_level")

display(department_representation.round(2))
display(job_level_representation.round(2))

,department,gender,gender_headcount,group_headcount,gender_share_pct
0,Data,Female,2403,4988,48.18
1,Data,Male,2428,4988,48.68
2,Data,Non-binary,58,4988,1.16
3,Data,Not recorded,20,4988,0.40
4,Data,Prefer not to say,79,4988,1.58
5,Engineering,Female,2425,5005,48.45
6,Engineering,Male,2435,5005,48.65
7,Engineering,Non-binary,53,5005,1.06
8,Engineering,Not recorded,19,5005,0.38
9,Engineering,Prefer not to say,73,5005,1.46


,job_level,gender,gender_headcount,group_headcount,gender_share_pct
0,1,Female,6090,12468,48.85
1,1,Male,5962,12468,47.82
2,1,Non-binary,136,12468,1.09
3,1,Not recorded,42,12468,0.34
4,1,Prefer not to say,238,12468,1.91
5,2,Female,7655,15933,48.04
6,2,Male,7753,15933,48.66
7,2,Non-binary,210,15933,1.32
8,2,Not recorded,41,15933,0.26
9,2,Prefer not to say,274,15933,1.72


In [22]:
hire_date = pd.to_datetime(
    standardised_pay_gap["hire_date"], errors="coerce"
).dt.normalize()

standardised_pay_gap["hire_year"] = hire_date.dt.year.astype("Int64")

In [23]:
snapshot_date = None
tolerance_years = 1

if snapshot_date is None:
    tenure_quality_summary = {
        "status": "Unable to check: snapshot date unavailable",
        "rows_checked": 0,
        "unable_to_check_rows": len(standardised_pay_gap),
    }
else:
    snapshot = pd.Timestamp(snapshot_date)
    recorded = pd.to_numeric(
        standardised_pay_gap["years_at_company"], errors="coerce"
    )
    expected = (snapshot - hire_date).dt.days / 365.25

    valid = hire_date.le(snapshot) & recorded.ge(0) & recorded.lt(float("inf"))
    difference = (recorded[valid] - expected[valid]).abs()

    tenure_quality_summary = {
        "status": "Check run",
        "snapshot_date": snapshot_date,
        "tolerance_years": tolerance_years,
        "rows_checked": len(difference),
        "consistent_rows": difference.le(tolerance_years).sum(),
        "inconsistent_rows": difference.gt(tolerance_years).sum(),
        "unable_to_check_rows": len(standardised_pay_gap) - len(difference),
    }

display(pd.DataFrame([tenure_quality_summary]))

,status,rows_checked,unable_to_check_rows
0,Unable to check: snapshot date unavailable,0,49880


In [24]:
from scipy.stats import ttest_ind

data = standardised_pay_gap
male_pay = data.loc[data["gender"].eq("Male"), "annual_base_pay_gbp"].dropna()
female_pay = data.loc[data["gender"].eq("Female"), "annual_base_pay_gbp"].dropna()

pay_test_summary = {
    "male_count": len(male_pay),
    "female_count": len(female_pay),
}

if min(len(male_pay), len(female_pay)) < 10:
    pay_test_summary["result"] = "Suppressed because a group has fewer than 10 employees"
else:
    mean_difference = male_pay.mean() - female_pay.mean()
    _, p_value = ttest_ind(male_pay, female_pay, equal_var=False)

    pay_test_summary["mean_difference_gbp"] = round(mean_difference, 2)
    pay_test_summary["p_value"] = p_value

    if pd.isna(p_value):
        pay_test_summary["result"] = "Unable to calculate a valid p-value"
    elif p_value < 0.05:
        pay_test_summary["result"] = "Statistically significant difference"
    else:
        pay_test_summary["result"] = "Insufficient evidence of a difference"

display(pd.DataFrame([pay_test_summary]))

,male_count,female_count,mean_difference_gbp,p_value,result
0,24074,24182,-84.43,0.56058,Insufficient evidence of a difference


In [25]:
from pathlib import Path
import pandas as pd

project_dir = (
    Path.home() / "Desktop" / "Projects"
    / "UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics"
)

input_data = pd.read_csv(project_dir / "processed" / "pay_gap_step1.csv")
accepted = standardised_pay_gap
rejected = pd.read_csv(project_dir / "rejected" / "rejected_pay_gap.csv")
warnings = pd.read_csv(project_dir / "processed" / "review_pay_gap_warnings.csv")

accepted_ids = accepted["source_row_number"]
rejected_ids = rejected["source_row_number"]
warning_ids = warnings["source_row_number"]

accounted_for = len(accepted) + len(rejected)

unique_row_numbers = all(
    ids.notna().all() and ids.is_unique
    for ids in [accepted_ids, rejected_ids, warning_ids]
)

reconciled = (
    accounted_for == len(input_data)
    and unique_row_numbers
    and not accepted_ids.isin(rejected_ids).any()
    and warning_ids.isin(accepted_ids).all()
)

report = pd.DataFrame({
    "row_category": [
        "Input rows",
        "Accepted without warnings",
        "Accepted with warnings",
        "Total accepted",
        "Rejected",
        "Total accounted for",
        "Input minus accounted for",
    ],
    "row_count": [
        len(input_data),
        (~accepted_ids.isin(warning_ids)).sum(),
        len(warnings),
        len(accepted),
        len(rejected),
        accounted_for,
        len(input_data) - accounted_for,
    ],
})

report["status"] = "Passed" if reconciled else "Failed"

documentation_dir = project_dir / "documentation"
documentation_dir.mkdir(exist_ok=True)
report.to_csv(documentation_dir / "pay_gap_reconciliation.csv", index=False)

display(report)

,row_category,row_count,status
0,Input rows,50000,Passed
1,Accepted without warnings,49472,Passed
2,Accepted with warnings,408,Passed
3,Total accepted,49880,Passed
4,Rejected,120,Passed
5,Total accounted for,50000,Passed
6,Input minus accounted for,0,Passed
